<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_05_02_random_forest_one2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_05_02 - ONE2ONE - Random Forest**

## **Introducción**

**Ridge Regression** es un modelo lineal regularizado que extiende la regresión lineal clásica (OLS) agregando una penalización sobre el tamaño de los coeficientes.

Su objetivo es:

* **Modelar relaciones lineales** entre features (alpha factors) y el target (ej. `delta_60`)
* **Reducir el overfitting** mediante regularización
* Servir como **baseline interpretable y estable**

Matemáticamente, minimiza:

$$
\min_{\beta} \left( \sum (y - X\beta)^2 + \lambda \sum \beta^2 \right)
$$

donde:

* $ \lambda $ controla la regularización
* Penaliza coeficientes grandes → modelo más estable

### **Intuición clave (importante para nuestro caso)**

En datasets financieros:

* Hay **ruido alto**
* Muchas features están **correlacionadas**
* La señal es **débil**

Ridge es útil porque:

* Reduce varianza sin eliminar variables (como Lasso)
* Maneja bien multicolinealidad
* Produce predicciones más **robustas OOS**

### **Rol en el pipeline**


En nuestro proyecto:

* Es el **primer modelo serio** después del baseline naïve
* Permite validar si:

  * Existe señal lineal en los features
  * Los alpha factors aportan valor real

Además, conecta directamente con el enfoque del libro, donde los modelos lineales regularizados son un punto de partida clave para predicción de retornos

# **Bloque común**

## **1. Imports + paths**

In [ ]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de variables X e y, y scalers**

In [ ]:
from pathlib import Path
import os
import pandas as pd
import joblib


XY_DELTA_DIR = DRIVE_DIR / Path(
    os.environ.get("XY_DELTA_DIR", "data/splits/")
)

XY_DELTA_DIR_SCALED = DRIVE_DIR / Path(
    os.environ.get("XY_DELTA_DIR_SCALED", "data/scaled/")
)

SCALERS_DIR = DRIVE_DIR / Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

TARGETS = ["delta_60", "delta_90"]
SPLITS = ["train", "valid", "test"]


def load_mnq_tabular_split(
    target: str,
    split: str,
    scaled: bool = False,
    return_scaler: bool = False,
):
    if target not in TARGETS:
        raise ValueError(f"target inválido: {target}. Esperados: {TARGETS}")

    if split not in SPLITS:
        raise ValueError(f"split inválido: {split}. Esperados: {SPLITS}")

    x_path = (
        XY_DELTA_DIR_SCALED / f"mnq_{target}_X_{split}_scaled.parquet"
        if scaled
        else XY_DELTA_DIR / f"mnq_{target}_X_{split}.parquet"
    )
    y_path = XY_DELTA_DIR / f"mnq_{target}_y_{split}.parquet"

    if not x_path.exists():
        raise FileNotFoundError(f"No existe X: {x_path}")
    if not y_path.exists():
        raise FileNotFoundError(f"No existe y: {y_path}")

    X = pd.read_parquet(x_path)
    y = pd.read_parquet(y_path)

    if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
        y = y.iloc[:, 0]

    if not return_scaler:
        return X, y

    scaler = None
    if scaled:
        scaler_path = SCALERS_DIR / f"scaler_{target}.pkl"
        if scaler_path.exists():
            scaler = joblib.load(scaler_path)

    return X, y, scaler

In [ ]:
#Sin escalado
SIN_ESCALADO = '''
X_train, y_train = load_mnq_tabular_split(
    target="delta_60",
    split="train",
    scaled=False,
)
'''

In [ ]:
#Escalado
ESCALADO = '''
X_train, y_train = load_mnq_tabular_split(
    target="delta_60",
    split="train",
    scaled=True,
)
'''


ESCALADO_ESCALADOR = '''
X_train, y_train, scaler = load_mnq_tabular_split(
    target="delta_60",
    split="train",
    scaled=True,
    load_scaler=True,
)
'''

In [ ]:
import pandas as pd
import numpy as np


def validate_tabular_dataset(
    X: pd.DataFrame,
    y: pd.Series | pd.DataFrame,
    *,
    name: str = "",
    check_index_alignment: bool = True,
    check_sorted: bool = True,
    date_col: str | None = None,
    verbose: bool = True,
):
    """
    Valida consistencia de un dataset tabular (X, y).

    Checks:
    - shapes
    - NaNs / inf
    - alineación de índices
    - orden temporal (opcional)
    - duplicados

    Retorna
    -------
    dict con flags de validación
    """

    report = {}

    # -------- Convertir y --------
    if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
        y = y.iloc[:, 0]

    # -------- Shapes --------
    report["n_samples_X"] = X.shape[0]
    report["n_samples_y"] = y.shape[0]
    report["n_features"] = X.shape[1]
    report["shape_match"] = X.shape[0] == y.shape[0]

    # -------- NaNs / inf --------
    report["X_has_nan"] = X.isna().any().any()
    report["y_has_nan"] = y.isna().any()

    report["X_has_inf"] = np.isinf(X.select_dtypes(include=[np.number])).any().any()
    report["y_has_inf"] = np.isinf(y).any()

    # -------- Índices --------
    if check_index_alignment:
        report["index_equal"] = X.index.equals(y.index)
    else:
        report["index_equal"] = None

    # -------- Orden temporal --------
    if check_sorted:
        if date_col and date_col in X.columns:
            report["sorted_by_date"] = X[date_col].is_monotonic_increasing
        else:
            report["sorted_by_index"] = X.index.is_monotonic_increasing
    else:
        report["sorted"] = None

    # -------- Duplicados --------
    report["duplicate_index"] = X.index.duplicated().any()

    # -------- Print --------
    if verbose:
        print(f"\n=== VALIDATION: {name} ===")
        for k, v in report.items():
            print(f"{k}: {v}")

        if not report["shape_match"]:
            print("⚠️ ERROR: X e y no tienen mismo número de filas")

        if report["X_has_nan"] or report["y_has_nan"]:
            print("⚠️ WARNING: Hay NaNs")

        if report["X_has_inf"] or report["y_has_inf"]:
            print("⚠️ WARNING: Hay valores infinitos")

        if check_index_alignment and not report["index_equal"]:
            print("⚠️ WARNING: Índices no alineados")

    return report

## **4. Reproducibilidad**

In [ ]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [ ]:
from pathlib import Path
import os
import sys
import importlib

DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

from metrics.one2one_metrics import evaluate_regression_predictions, print_metrics

print("OK - imports metrics.*")

import metrics.one2one_metrics as m

#print(m.__doc__)
#print(m.evaluate_regression_predictions.__doc__)


OK - imports metrics.*


## **6. Métricas Machine Learning**

In [ ]:
import pandas as pd


def evaluate_model_on_split(
    model,
    X,
    y,
    *,
    split_name: str,
    model_name: str,
    target_name: str,
):
    """
    Evalúa un modelo sobre un split dado y devuelve:
    - y_pred
    - dict de métricas
    """
    y_pred = model.predict(X)

    metrics = evaluate_regression_predictions(
        y_true=y,
        y_pred=y_pred,
        split_name=split_name,
        model_name=model_name,
        target_name=target_name,
    )

    return y_pred, metrics


def metrics_to_df(metrics: dict) -> pd.DataFrame:
    """
    Convierte un dict de métricas en una fila de DataFrame.
    Versión final sin redundancias (ni window_size ni horizon).
    """
    row = {
        "model": metrics.get("model"),
        "split": metrics.get("split"),
        "target": metrics.get("target"),
        "n_samples": metrics.get("n_samples"),
        "mae": metrics.get("mae"),
        "rmse": metrics.get("rmse"),
        "r2": metrics.get("r2"),
        "directional_accuracy": metrics.get("directional_accuracy"),
    }

    return pd.DataFrame([row])

def evaluate_model_on_bundle(
    model,
    bundle: dict,
    *,
    model_name: str,
    target_name: str,
    window_size: int | None = None,
    horizon: int | None = None,
    splits: tuple[str, ...] = ("valid", "test"),
):
    """
    Evalúa un modelo en varios splits de un bundle.

    Estructura esperada de bundle:
    bundle = {
        "train": {"X": ..., "y": ...},
        "valid": {"X": ..., "y": ...},
        "test":  {"X": ..., "y": ...},
    }

    Retorna
    -------
    predictions : dict
        Predicciones por split.
    metrics_dict : dict
        Métricas por split.
    metrics_df : pd.DataFrame
        Tabla consolidada.
    """
    predictions = {}
    metrics_dict = {}
    frames = []

    for split in splits:
        X = bundle[split]["X"]
        y = bundle[split]["y"]

        y_pred, metrics = evaluate_model_on_split(
            model=model,
            X=X,
            y=y,
            split_name=split,
            model_name=model_name,
            target_name=target_name,
        )

        predictions[split] = y_pred
        metrics_dict[split] = metrics
        frames.append(
            metrics_to_df(
                metrics,
                window_size=window_size,
                horizon=horizon,
            )
        )

    metrics_df = pd.concat(frames, ignore_index=True)

    return predictions, metrics_dict, metrics_df

## **7. Gestión de dataset de métricas**

In [ ]:
def load_one2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/one2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"one2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [ ]:
from pathlib import Path
import pandas as pd

def save_one2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/one2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"one2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

# **DEFINICIÓN DE MODELO**

## **8. Definición del modelo — placeholder**

### **8.1. Modelo Ridge Regression (one2one)**

**Idea básica**

**Ridge Regression** es una regresión lineal con **regularización L2**.
Aprende una relación lineal entre las features de entrada y un target escalar, penalizando coeficientes grandes para reducir sobreajuste y mejorar estabilidad fuera de muestra.

Formalmente:

$$
\hat{y}_i = \mathbf{w}^\top \mathbf{x}_i + b
$$

donde:

* $\mathbf{x}_i$: vector de features de la observación (i)
* $\mathbf{w}$: coeficientes del modelo
* $b$: intercepto

La función objetivo es:

$$
\min_{\mathbf{w}, b}
\sum_i (y_i - \hat{y}_i)^2
;+;
\alpha \sum_j w_j^2
$$

donde $\alpha$ controla la **fuerza de la regularización**.

---

**Regularización (Ridge vs Lasso)**

* **Ridge (L2)**
  Penaliza el cuadrado de los coeficientes.
  Tiende a reducir su magnitud, pero normalmente **no los lleva a cero**.

* **Lasso (L1)**
  Penaliza el valor absoluto de los coeficientes.
  Puede llevar algunos pesos exactamente a cero, actuando también como selector de variables.

En este caso usamos **Ridge** porque prioriza **estabilidad**, especialmente cuando hay multicolinealidad entre variables.

---

**Por qué Ridge encaja bien en este proyecto**

Ridge es una muy buena primera referencia para este problema porque:

* trabaja bien con **features numéricas tabulares**
* tolera mejor la **multicolinealidad**
* es **rápido de entrenar**
* es **fácil de interpretar**
* ofrece un baseline lineal y regularizado antes de pasar a modelos más complejos

En problemas financieros, donde la señal suele ser débil y ruidosa, un modelo lineal regularizado permite verificar si existe al menos una señal explotable en las variables construidas.

---

**Rol dentro del pipeline**

En este proyecto, Ridge cumple el rol de:

* **baseline entrenable**
* referencia contra la cual comparar:

  * Lasso
  * Random Forest
  * XGBoost / LightGBM
  * MLP
  * LSTM / Transformer

Si Ridge ya logra una señal razonable en validación y test, entonces hay evidencia de que las features contienen información útil. Si no lo logra, eso también es informativo.

---

**Hiperparámetros iniciales**

Para esta primera versión del modelo:

* `alpha = 1.0`
* `fit_intercept = True`

No requiere:

* `dropout`
* `early stopping`
* entrenamiento por épocas

porque es un modelo lineal cerrado/convexo, mucho más simple que una red neuronal.

---

**Nota para este stage**

En esta etapa, el objetivo no es todavía optimizar Ridge al máximo, sino usarlo como una referencia sólida para responder:

* si existe señal lineal en las features
* qué tan competitivo es un modelo simple
* cuánto valor adicional aportan luego los modelos no lineales

### **8.2. Carga de X e y**

In [ ]:
TARGETS = ["delta_60", "delta_90"]

data = {}

for target in TARGETS:
    print(f"\n==============================")
    print(f"CARGANDO DATASET: {target}")
    print(f"==============================")

    # -------- TRAIN --------
    X_train, y_train, scaler = load_mnq_tabular_split(
        target=target,
        split="train",
        scaled=True,
        return_scaler=True,
    )

    validate_tabular_dataset(
        X_train,
        y_train,
        name=f"train_{target}",
    )

    # -------- VALID --------
    X_valid, y_valid = load_mnq_tabular_split(
        target=target,
        split="valid",
        scaled=True,
    )

    validate_tabular_dataset(
        X_valid,
        y_valid,
        name=f"valid_{target}",
    )

    # -------- TEST --------
    X_test, y_test = load_mnq_tabular_split(
        target=target,
        split="test",
        scaled=True,
    )

    validate_tabular_dataset(
        X_test,
        y_test,
        name=f"test_{target}",
    )

    # -------- Guardar --------
    data[target] = {
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
        "scaler": scaler,
    }


CARGANDO DATASET: delta_60

=== VALIDATION: train_delta_60 ===
n_samples_X: 490146
n_samples_y: 490146
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_index: False

=== VALIDATION: valid_delta_60 ===
n_samples_X: 104954
n_samples_y: 104954
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_index: False

=== VALIDATION: test_delta_60 ===
n_samples_X: 105495
n_samples_y: 105495
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_index: False

CARGANDO DATASET: delta_90

=== VALIDATION: train_delta_90 ===
n_samples_X: 490146
n_samples_y: 490146
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_in

In [ ]:
X_train_delta_60 = data["delta_60"]["train"]["X"]
y_train_delta_60 = data["delta_60"]["train"]["y"]

X_valid_delta_60 = data["delta_60"]["valid"]["X"]
y_valid_delta_60 = data["delta_60"]["valid"]["y"]

X_test_delta_60  = data["delta_60"]["test"]["X"]
y_test_delta_60  = data["delta_60"]["test"]["y"]

X_train_delta_90 = data["delta_90"]["train"]["X"]
y_train_delta_90 = data["delta_90"]["train"]["y"]

X_valid_delta_90 = data["delta_90"]["valid"]["X"]
y_valid_delta_90 = data["delta_90"]["valid"]["y"]

X_test_delta_90  = data["delta_90"]["test"]["X"]
y_test_delta_90  = data["delta_90"]["test"]["y"]

### **8.3. Entrenamiento**

In [ ]:
from sklearn.linear_model import Ridge


def train_evaluate_ridge_one2one(
    *,
    target_name: str,
    X_train,
    y_train,
    X_valid,
    y_valid,
    X_test,
    y_test,
    alpha: float = 1.0,
    fit_intercept: bool = True,
):
    """
    Entrena y evalúa un Ridge Regression one-to-one para un target dado.

    Retorna
    -------
    results : dict
        Contiene modelo, predicciones y métricas por split.
    """
    model = Ridge(
        alpha=alpha,
        fit_intercept=fit_intercept,
    )

    # -------------------------
    # Entrenamiento
    # -------------------------
    model.fit(X_train, y_train)

    # -------------------------
    # Predicciones
    # -------------------------
    y_pred_valid = model.predict(X_valid)
    y_pred_test = model.predict(X_test)

    # -------------------------
    # Métricas
    # -------------------------
    metrics_valid = evaluate_regression_predictions(
        y_true=y_valid,
        y_pred=y_pred_valid,
        split_name="valid",
        model_name="ridge",
        target_name=target_name,
    )

    metrics_test = evaluate_regression_predictions(
        y_true=y_test,
        y_pred=y_pred_test,
        split_name="test",
        model_name="ridge",
        target_name=target_name,
    )

    results = {
        "model": model,
        "target": target_name,
        "predictions": {
            "valid": y_pred_valid,
            "test": y_pred_test,
        },
        "metrics": {
            "valid": metrics_valid,
            "test": metrics_test,
        },
    }

    return results

In [ ]:
ridge_delta_60 = train_evaluate_ridge_one2one(
    target_name="delta_60",
    X_train=X_train_delta_60,
    y_train=y_train_delta_60,
    X_valid=X_valid_delta_60,
    y_valid=y_valid_delta_60,
    X_test=X_test_delta_60,
    y_test=y_test_delta_60,
    alpha=1.0,
    fit_intercept=True,
)

print_metrics(ridge_delta_60["metrics"]["valid"])
print_metrics(ridge_delta_60["metrics"]["test"])

=== Regression Metrics ===
model: ridge
target: delta_60
split: valid
n_samples: 104954
mae: 34.254025
rmse: 49.172692
r2: -0.002405
directional_accuracy: 0.497866
=== Regression Metrics ===
model: ridge
target: delta_60
split: test
n_samples: 105495
mae: 52.021664
rmse: 81.608383
r2: 0.000442
directional_accuracy: 0.499313


In [ ]:
ridge_delta_60["metrics"]["valid"]

{'model': 'ridge',
 'target': 'delta_60',
 'split': 'valid',
 'n_samples': 104954,
 'mae': 34.25402467132122,
 'rmse': 49.17269163053438,
 'r2': -0.0024048015301194603,
 'directional_accuracy': 0.4978657316538674}

In [ ]:
ridge_delta_90 = train_evaluate_ridge_one2one(
    target_name="delta_90",
    X_train=X_train_delta_90,
    y_train=y_train_delta_90,
    X_valid=X_valid_delta_90,
    y_valid=y_valid_delta_90,
    X_test=X_test_delta_90,
    y_test=y_test_delta_90,
    alpha=1.0,
    fit_intercept=True,
)

print_metrics(ridge_delta_90["metrics"]["valid"])
print_metrics(ridge_delta_90["metrics"]["test"])

=== Regression Metrics ===
model: ridge
target: delta_90
split: valid
n_samples: 104954
mae: 42.916488
rmse: 61.539001
r2: -0.002588
directional_accuracy: 0.499533
=== Regression Metrics ===
model: ridge
target: delta_90
split: test
n_samples: 105495
mae: 65.586614
rmse: 101.784534
r2: 0.001354
directional_accuracy: 0.495843


### **8.4. Guardado de datasets de métricas**

In [ ]:
# =========================================================
# 1. DELTA 60
# =========================================================

df_valid_60 = metrics_to_df(
    ridge_delta_60["metrics"]["valid"]
)

df_test_60 = metrics_to_df(
    ridge_delta_60["metrics"]["test"]
)

df_ridge_60 = pd.concat([df_valid_60, df_test_60], ignore_index=True)


# =========================================================
# 2. DELTA 90
# =========================================================

df_valid_90 = metrics_to_df(
    ridge_delta_90["metrics"]["valid"]
)

df_test_90 = metrics_to_df(
    ridge_delta_90["metrics"]["test"]
)

df_ridge_90 = pd.concat([df_valid_90, df_test_90], ignore_index=True)


# =========================================================
# 3. CONSOLIDACIÓN
# =========================================================

df_all_ridge = pd.concat([df_ridge_60, df_ridge_90], ignore_index=True)


# =========================================================
# 4. GUARDADO
# =========================================================

save_one2one_metrics(
    df_all_ridge,
    name="ridge_all",
)

[OK] Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/one2one_metrics/one2one_ridge_all_metrics.parquet


PosixPath('/content/drive/MyDrive/neural_profit/metrics/one2one_metrics/one2one_ridge_all_metrics.parquet')

In [ ]:
df_all_ridge

,model,split,target,n_samples,mae,rmse,r2,directional_accuracy
0,ridge,valid,delta_60,104954,34.254025,49.172692,-0.002405,0.497866
1,ridge,test,delta_60,105495,52.021664,81.608383,0.000442,0.499313
2,ridge,valid,delta_90,104954,42.916488,61.539001,-0.002588,0.499533
3,ridge,test,delta_90,105495,65.586614,101.784534,0.001354,0.495843
